# 10 — Synthetic Anomaly Stress Testing

## Objective
Inject controlled synthetic anomalies (e.g. extreme purchase values, impossible account ages, high-sharing devices) and verify that the ranking model surfaces them at the top.

**Important:** Original raw data is never modified; we work on a copy.


In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
from data_utils import load_processed
from anomaly import fit_isolation_forest
from evaluation import evaluate_anomaly_scores

ROOT = Path.cwd()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

X_train = load_processed("X_train").drop(columns=["class"])
X_val   = load_processed("X_val").drop(columns=["class"]).copy()

# Inject 50 synthetic anomalies
rng = np.random.RandomState(42)
n_syn = 50
syn = X_val.sample(n_syn, random_state=42).copy()
if "purchase_value" in syn.columns:
    syn["purchase_value"] = syn["purchase_value"] * 10 + 500
if "account_age_hours" in syn.columns:
    syn["account_age_hours"] = 0.0001
if "users_on_device" in syn.columns:
    syn["users_on_device"] = 50

X_stress = pd.concat([X_val, syn], ignore_index=True)
y_stress = np.array([0]*len(X_val) + [1]*n_syn)

model = fit_isolation_forest(X_train, contamination=0.09)
scores = model.predict_anomaly_score(X_stress)
metrics = evaluate_anomaly_scores(y_stress, scores)
print("Synthetic anomaly recovery metrics:")
print(metrics)
print("\nTop-10 scores among synthetic:", sorted(scores[-n_syn:], reverse=True)[:10])
